<a href="https://colab.research.google.com/github/doanthuchuyen/thuchuyen07.github.io/blob/gh-pages/BT04_ex2_MissingValue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer # KNNImputer là một thay đổi lớn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression # Sử dụng Logistic Regression thay cho Random Forest
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [2]:
try:
    df_titanic = sns.load_dataset('titanic')
except ValueError:
    # Nếu không tải được từ seaborn, tạo DataFrame giả lập có các cột tương tự
    data = {
        'age': np.random.choice([20, 30, np.nan], size=891),
        'fare': np.random.rand(891) * 100,
        'pclass': np.random.randint(1, 4, size=891),
        'sex': np.random.choice(['male', 'female'], size=891),
        'embarked': np.random.choice(['S', 'C', 'Q', np.nan], size=891, p=[0.5, 0.2, 0.2, 0.1]),
        'survived': np.random.randint(0, 2, size=891)
    }
    df_titanic = pd.DataFrame(data)

In [3]:
features = ['pclass', 'sex', 'age', 'fare', 'embarked']
target = 'survived'
df = df_titanic[features + [target]].copy()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=0.3, random_state=42
)

In [5]:
numerical_features = ['age', 'fare']

In [6]:
categorical_features = ['pclass', 'sex', 'embarked']

In [7]:
numerical_transformer_knn = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler())
])

In [8]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [9]:
preprocessor_knn = ColumnTransformer(
    transformers=[
        ('num_knn', numerical_transformer_knn, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

In [10]:
pipeline_knn = Pipeline(steps=[
    ('preprocessor', preprocessor_knn),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42))
])

In [11]:
pipeline_knn.fit(X_train, y_train)
y_pred_knn = pipeline_knn.predict(X_test)
score_knn = f1_score(y_test, y_pred_knn)

print("--- Đánh giá mô hình sau khi điền khuyết bằng KNN Imputer ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"F1 Score: {score_knn:.4f}")
print("-" * 50)

--- Đánh giá mô hình sau khi điền khuyết bằng KNN Imputer ---
Accuracy: 0.8060
F1 Score: 0.7500
--------------------------------------------------


In [12]:
numerical_transformer_simple = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

In [13]:
preprocessor_simple = ColumnTransformer(
    transformers=[
        ('num_simple', numerical_transformer_simple, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

In [14]:
pipeline_simple = Pipeline(steps=[
    ('preprocessor', preprocessor_simple),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42))
])

In [15]:
pipeline_simple.fit(X_train, y_train)
y_pred_simple = pipeline_simple.predict(X_test)
score_simple = f1_score(y_test, y_pred_simple)

print("--- Đánh giá mô hình sau khi điền khuyết bằng Mean Imputer ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_simple):.4f}")
print(f"F1 Score: {score_simple:.4f}")
print("-" * 50)

--- Đánh giá mô hình sau khi điền khuyết bằng Mean Imputer ---
Accuracy: 0.8022
F1 Score: 0.7464
--------------------------------------------------


In [16]:
results = pd.DataFrame({
    'Phương pháp': ['KNN Imputer', 'Mean Imputer'],
    'F1 Score': [score_knn, score_simple]
})

print("Tóm tắt so sánh các phương pháp điền khuyết:")
display(results)

Tóm tắt so sánh các phương pháp điền khuyết:


,Phương pháp,F1 Score
0,KNN Imputer,0.750000
1,Mean Imputer,0.746411
